- run on maxg16
- conda env: scib_metrics

In [1]:
# computed on donor_id and author_cell_type

In [1]:
import scanpy as sc
import anndata as ad
import scib
import scib_metrics
import numpy as np
import pandas as pd

/fast/AG_Ohler/prauten/conda_envs/scib_metrics/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/fast/AG_Ohler/prauten/conda_envs/scib_metrics/lib/python3.10/site-packages/umap/__init__.py:9: ImportWarning: Tensorflow not installed; ParametricUMAP will be unavailable
  warn(


In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=ad.OldFormatWarning)

In [3]:
scenarios = ['integration_hbca', 'noIntegration_hbca', 'naiveIntegration_donor_id_hbca']

In [4]:
# use all the "fast" metrics
np.random.seed(61)

# Collect computed scores, nested dict is simple to convert to pd.DataFrame
score_dict = {}
for scenario in scenarios:
    # Initialize nested dict
    score_dict[scenario] = {}
    
    adata = ad.read_h5ad('./../../embeddings/{}.embedding.h5ad'.format(scenario))
    adata.obsm['embedding'] = adata.X
    
    sc.pp.neighbors(adata, use_rep='embedding')

    # Compute scores
    ## Level of evaluation: batch/sample
    ### asw_batch
    score = scib_metrics.metrics.silhouette_batch(
        adata.obsm['embedding'],
        adata.obs['author_cell_type'],
        adata.obs['donor_id']
    )
    score_dict[scenario]['asw_batch'] = score

    score = scib_metrics.metrics.silhouette_batch(
        adata.obsm['embedding'],
        adata.obs['author_cell_type'],
        adata.obs['donor_id'],
        metric='cosine'
    )
    score_dict[scenario]['asw_batch_cosine'] = score
    
    ### bras aka. asw_batch_mean_other
    score = scib_metrics.metrics.bras(
        adata.obsm['embedding'],
        adata.obs['author_cell_type'],
        adata.obs['donor_id']
    )
    score_dict[scenario]['bras'] = score

    score = scib_metrics.metrics.bras(
        adata.obsm['embedding'],
        adata.obs['author_cell_type'],
        adata.obs['donor_id'],
        metric='euclidean'
    )
    score_dict[scenario]['bras_euclidean'] = score

    # bras_furthest aka. asw_batch_furthest
    score = scib_metrics.metrics.bras(
        adata.obsm['embedding'],
        adata.obs['author_cell_type'],
        adata.obs['donor_id'],
        between_cluster_distances='furthest'
        
    )
    score_dict[scenario]['bras_furthest'] = score

    score = scib_metrics.metrics.bras(
        adata.obsm['embedding'],
        adata.obs['author_cell_type'],
        adata.obs['donor_id'],
        between_cluster_distances='furthest',
        metric='euclidean'
    )
    score_dict[scenario]['bras_furthest_euclidean'] = score

    ### asw_label
    score = scib_metrics.metrics.silhouette_label(
        adata.obsm['embedding'],
        adata.obs['author_cell_type']
    )
    score_dict[scenario]['asw_label'] = score

    score = scib_metrics.metrics.silhouette_label(
        adata.obsm['embedding'],
        adata.obs['author_cell_type'],
        metric='cosine'
    )
    score_dict[scenario]['asw_label_cosine'] = score

    ### graph iLISI and cLISI on variable batch
    score_dict[scenario]['iLISI_batch'], score_dict[scenario]['cLISI_full'] =  scib.me.lisi.lisi_graph(adata, batch_key='donor_id', label_key='author_cell_type', type_='knn')

    ## CiLISI
    means = []
    total = 0
    for cell_type in adata.obs['author_cell_type'].unique():
        tmp_adata = adata[adata.obs['author_cell_type']==cell_type]
        cell_type_iLISI = scib.metrics.ilisi_graph(tmp_adata, batch_key='donor_id', type_='knn')
        means += [cell_type_iLISI * tmp_adata.shape[0]]
        total += tmp_adata.shape[0]
        print(cell_type, cell_type_iLISI)
    print(means)
    print(np.nansum(means)/total)
    score_dict[scenario]['CiLISI_batch'] = np.nansum(means)/total
    
    ### nmi and ari  
    neigh_result_90 = scib_metrics.nearest_neighbors.pynndescent(
                    adata.obsm['embedding'], n_neighbors=90, random_state=0)# , n_jobs=self._n_jobs)
    nmi_ari_dict = scib_metrics.nmi_ari_cluster_labels_leiden(
        neigh_result_90,
        adata.obs['author_cell_type'],
    )

    score_dict[scenario]['nmi'], score_dict[scenario]['ari'] = nmi_ari_dict['nmi'], nmi_ari_dict['ari']

LASP 0.3072938818758176
Adi-2 0.3616404148857703
Chunk 4250 does not have enough neighbors. Skipping...
Chunk 4376 does not have enough neighbors. Skipping...
Chunk 4435 does not have enough neighbors. Skipping...
Chunk 4557 does not have enough neighbors. Skipping...
Fibroblasts 0.31456008231786736
Adi-1 0.2961330583306934
T-cells 0.2935756300104418
Endo-1 0.28495808906353304
BM 0.3276406129063538
Chunk 244 does not have enough neighbors. Skipping...
Chunk 1299 does not have enough neighbors. Skipping...
Chunk 1539 does not have enough neighbors. Skipping...
Macrophages 0.26496530004082713
LHS 0.320423424219545
Endo-2 0.3135810672773846
[3520.358710769366, 194.20090279365866, 2052.8190972064026, 644.0894018692582, 657.6094112233897, 1057.1945104257077, 5302.5356792764305, 434.5430920669565, 1666.842652790073, 532.1470711697217]
0.3126976566587686
Chunk 241 does not have enough neighbors. Skipping...
Chunk 2103 does not have enough neighbors. Skipping...
Chunk 8008 does not have enough

In [5]:
scores = pd.DataFrame(score_dict)

In [6]:
scores

,integration_hbca,noIntegration_hbca,naiveIntegration_donor_id_hbca
asw_batch,0.846903,0.878997,0.868476
asw_batch_cosine,0.712453,0.769370,0.783068
bras,0.864901,0.694869,0.765132
bras_euclidean,0.923914,0.803634,0.843235
bras_furthest,0.680168,0.453299,0.548777
bras_furthest_euclidean,0.809701,0.525383,0.640061
asw_label,0.676198,0.605599,0.627097
asw_label_cosine,0.794168,0.671227,0.714133
iLISI_batch,0.314805,0.113782,0.190549
cLISI_full,1.000000,0.999512,1.000000


In [7]:
pd.DataFrame(score_dict).to_csv("./../../evaluation/batch_removal_scores_real_data_hbca.csv", index=True)